In [63]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..","DTENV"))
Pkg.instantiate()
if size(LOAD_PATH,1) < 4
    push!(LOAD_PATH, joinpath(@__DIR__,"..","scripts"))
end

  Activating project at `~/Blk/JuliaDTFE/DTENV`


In [64]:
using TetGen
using StaticArrays
using JLD
using BenchmarkTools
using LinearAlgebra
using Plots

In [65]:
include("../scripts/TesselationCore.jl")
import .TesselationCore

BVH = TesselationCore.BVH
point3 = TesselationCore.point3

SVector{3, Float64} (alias for SArray{Tuple{3}, Float64, 1, 3})

In [66]:
import illustris_julia as il


basePath = "../../ThesisMaster/Illustris/";

fields = ["SubhaloMass","SubhaloCM"];
subhalos = il.groupcat.loadSubhalos(basePath,135,fields)

positions = subhalos["SubhaloCM"]
print("done") #otherwise floods GitHub with output 

done

In [67]:
gap = 500
points = positions[:,1:gap:end]
ps = [point3(points[1,i], points[2,i], points[3,i]) for i in 1:size(points,2)]

bvh,tes,tets = TesselationCore.standardEstimator(ps,10)
print("done")

done

In [68]:
N = 1000

width = 75000

step = width/N


xs = bvh.bbox[1,1]:step:bvh.bbox[1,2]
ys = bvh.bbox[2,1]:step:bvh.bbox[2,2]

z = (bvh.bbox[3,2] + bvh.bbox[3,1])/2

dens = zeros(N,N)
print("Done")

Done

In [69]:
typeof(tes.points[tets])

Matrix{SVector{3, Float64}} (alias for Array{SArray{Tuple{3}, Float64, 1, 3}, 2})

In [70]:
typeof(pts)

Vector{Vector{Float64}} (alias for Array{Array{Float64, 1}, 1})

In [71]:
pts = [[xs[i],ys[i],z] for i in [5,10,20,30,50,100]]
#simps = tes.points[tets]
#ids = [TesselationCore.findID(points[:,i], tes.points[tets], bvh) for i in 1:size(points,2)]

TesselationCore.DTFE(pts,bvh,tets,tes)

MethodError: MethodError: no method matching isless(::Vector{Float64}, ::Float64)
The function `isless` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  isless(!Matched::Missing, ::Any)
   @ Base missing.jl:87
  isless(::Any, !Matched::Missing)
   @ Base missing.jl:88
  isless(!Matched::T, ::T) where T<:Union{Float16, Float32, Float64}
   @ Base float.jl:630
  ...


In [11]:
for (i,x) in pairs(xs)
    for (j,y) in pairs(ys)
        dens[i,j] = DTFE([x,y,z],bvh,tets,tes)

    end
end

UndefVarError: UndefVarError: `xs` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [12]:
med = median(dens)

UndefVarError: UndefVarError: `dens` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [13]:
Plots.heatmap(dens ./med,clim=(0,25))

UndefVarError: UndefVarError: `dens` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [12]:
N = 100

width = 75000

step = width/N


xs = bvh.bbox[1,1]:step:bvh.bbox[1,2]
ys = bvh.bbox[2,1]:step:bvh.bbox[2,2]

zs =  bvh.bbox[3,1]:step:bvh.bbox[3,2]

dens = zeros(N,N,N)
print("Done")

Done

In [13]:
DTFE([35000.,35000.,35000.],bvh,tets,tes)

2.1390745266731447e-12

In [14]:
for (i,x) in pairs(xs)
    for (j,y) in pairs(ys)
        for (k,z) in pairs(zs)
            dens[i,j,k] = DTFE([x,y,z],bvh,tets,tes)

        end

    end
end

In [15]:
using ColorSchemes
using GLMakie
lowColor  = get(ColorSchemes.acton,LinRange(0,1,256))[1]

fig = GLMakie.Figure(size = (1600,1600),backgroundcolor=lowColor)
ax = GLMakie.LScene(fig[1,1],scenekw=(show_axis=false,backgroundcolor=lowColor))
volplot = volume!(
    ax,dens ./median(dens),
    algorithm=:mip,
    colormap = :acton,
    colorrange = (.0,25),
    )

Volume{Tuple{Makie.EndPoints{Float32}, Makie.EndPoints{Float32}, Makie.EndPoints{Float32}, Array{Float32, 3}}}

In [16]:
fig

In [19]:
save("../Images/DTFE.png", fig)  